# 02 · Vectorization — Doc2Vec vs Sentence-BERT

Two text vectorizers sitting at opposite ends of the spectrum:

| Method | Family | Trained on | Output dim |
|---|---|---|---|
| Doc2Vec (PV-DM) | shallow self-supervised | this corpus only (~750 docs) | 200 |
| Sentence-BERT (MiniLM-multilingual) | pretrained Transformer | hundreds of millions of sentence pairs | 384 |

Both methods feed Ward (notebook 03) and the comparison answers an empirical question: does training on this small corpus beat a frozen multilingual encoder? Both produce dense learned embeddings, so the comparison is genuinely apples-to-apples — depth of training and breadth of pre-training corpus are the only differing axes.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, logging, json, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(message)s')

from src.config import VECTORS_DIR, CLUSTERS_DIR, PROCESSED_DIR
corpus = pd.read_parquet(PROCESSED_DIR / 'corpus.parquet')
print('corpus:', corpus.shape, '— sources:', corpus['source'].value_counts().to_dict())

## 2.1 Doc2Vec — train on this corpus
PV-DM, 200-D, 40 epochs. Uses the `tokens` column already produced by jieba in `preprocess.build_corpus`.

In [ ]:
from src.vectorize import embed_doc2vec
from src.vectorize.compare import time_call

text_corpus = corpus[corpus['text'].str.len() > 0].reset_index(drop=True)
ids = text_corpus['id'].tolist()
token_lists = [t.split() if isinstance(t, str) else [] for t in text_corpus['tokens']]

doc2vec_artefact, doc2vec_seconds = time_call(embed_doc2vec, ids, token_lists)
print('Doc2Vec:', doc2vec_artefact.embeddings.shape, 'fit_seconds=', round(doc2vec_seconds, 2))

## 2.2 Sentence-BERT — frozen pretrained inference
Multilingual MiniLM. Zero training; first call downloads the ~120 MB model from HuggingFace.

In [ ]:
from src.vectorize import embed_sbert
sbert_artefact, sbert_seconds = time_call(
    embed_sbert, ids, text_corpus['text'].tolist(),
)
print('Sentence-BERT:', sbert_artefact.embeddings.shape, 'fit_seconds=', round(sbert_seconds, 2))

## 2.3 Empirical comparison
Doc2Vec embeddings are L2-normalised inside the comparator; SBERT vectors are already unit-normalised.

In [ ]:
from src.vectorize import compare_methods
report = compare_methods(
    doc2vec_embeddings=doc2vec_artefact.embeddings,
    sbert_embeddings=sbert_artefact.embeddings,
    sources=text_corpus['source'].tolist(),
    doc2vec_seconds=doc2vec_seconds,
    sbert_seconds=sbert_seconds,
)
CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
(CLUSTERS_DIR / 'comparison.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
report

## 2.4 Pros & cons — based on these numbers, not theory
(populated above; the comparator's `notes` field already tells the empirical story).

In [ ]:
for k, r in report.items():
    print(f"\n=== {r['method']} ===")
    print(f"  silhouette                 : {r['silhouette']:.4f}")
    print(f"  top-5 source purity         : {r['mean_intra_source_topk_purity']:.4f}")
    print(f"  fit time (seconds)          : {r['fit_seconds']:.2f}")
    print('  notes:')
    for n in r['notes']:
        print('   -', n)

## 2.5 Save the embeddings
Both artefacts are persisted to `data/vectors/`. Notebook 03 picks the SBERT vectors as the primary signal (Doc2Vec is kept for the comparison only).

In [ ]:
doc2vec_artefact.save(VECTORS_DIR / 'doc2vec.npz')
sbert_artefact.save(VECTORS_DIR / 'sbert.npz')
print('saved →', VECTORS_DIR)